In [1]:
import pandas as pd
import numpy as np

In [2]:
features_df = pd.read_csv('data/features_processed.csv')
features_df.head()

,date,home_team,away_team,home_GF5,home_GA5,home_W5,home_D5,home_L5,home_points5,home_mean_competition5,...,home_unbeaten_streak,away_unbeaten_streak,h2h_home_wins,h2h_away_wins,h2h_draws,h2h_matches,neutral,competition_level,home_score,away_score
0,2000-01-04,Egypt,Togo,0.0,0.0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,1,2,1
1,2000-01-07,Tunisia,Togo,0.0,0.0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,1,7,0
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,1,0,0
3,2000-01-09,Mexico,Iran,0.0,0.0,0,0,0,0,0.0,...,0,0,0,0,0,0,1,1,2,1
4,2000-01-09,Ivory Coast,Egypt,0.0,0.0,0,0,0,0,0.0,...,0,1,0,0,0,0,0,1,2,0


In [3]:
features_df["date"] = pd.to_datetime(features_df["date"])

In [4]:
selected_features_m4 = [
    # Últimos 10 partidos
    "home_GF10",
    "home_GA10",
    "home_points10",
    "home_mean_competition10",

    "away_GF10",
    "away_GA10",
    "away_points10",
    "away_mean_competition10",

    # Elo
    "home_elo",
    "away_elo",
    "elo_diff",

    # H2H
    "h2h_home_wins",
    "h2h_away_wins",
    "h2h_draws",
    "h2h_matches",

    # Contexto
    "neutral",
    "competition_level"
]

X_m4 = features_df[selected_features_m4]

y_home = features_df["home_score"]
y_away = features_df["away_score"]

In [5]:
features_df["date"] = pd.to_datetime(features_df["date"])

train_mask = features_df["date"] < "2023-01-01"

val_mask = (
    (features_df["date"] >= "2023-01-01") &
    (features_df["date"] < "2025-01-01")
)

test_mask = features_df["date"] >= "2025-01-01"

X_train = X_m4[train_mask]
X_val = X_m4[val_mask]
X_test = X_m4[test_mask]

y_home_train = y_home[train_mask]
y_home_val = y_home[val_mask]
y_home_test = y_home[test_mask]

y_away_train = y_away[train_mask]
y_away_val = y_away[val_mask]
y_away_test = y_away[test_mask]

In [6]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, accuracy_score

In [7]:


param_grid = {
    "n_estimators": [200, 300, 500, 700, 1000],
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7, 10],
    "gamma": [0, 0.05, 0.1, 0.2, 0.5],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [0.5, 1, 2, 5]
}

In [ ]:
home_base = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

home_search = RandomizedSearchCV(
    estimator=home_base,
    param_distributions=param_grid,
    n_iter=50,
    scoring="neg_mean_absolute_error",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

home_search.fit(X_train, y_home_train)

print("Mejores parámetros HOME:")
print(home_search.best_params_)
print("Mejor MAE CV HOME:", home_search.best_score_)

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Mejores parámetros HOME:
{'subsample': 0.8, 'reg_lambda': 0.5, 'reg_alpha': 0.01, 'n_estimators': 700, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 1.0}
Mejor MAE CV HOME: 1.0808000167210896


In [9]:
away_base = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

away_search = RandomizedSearchCV(
    estimator=away_base,
    param_distributions=param_grid,
    n_iter=50,
    scoring="neg_mean_absolute_error",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

away_search.fit(X_train, y_away_train)

print("Mejores parámetros AWAY:")
print(away_search.best_params_)
print("Mejor MAE CV AWAY:", -away_search.best_score_)

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Mejores parámetros AWAY:
{'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 10, 'max_depth': 4, 'learning_rate': 0.03, 'gamma': 0.05, 'colsample_bytree': 0.6}
Mejor MAE CV AWAY: 0.8723123669624329


In [10]:
home_model_opt = home_search.best_estimator_
away_model_opt = away_search.best_estimator_

home_pred_val = home_model_opt.predict(X_val)
away_pred_val = away_model_opt.predict(X_val)

mae_home = mean_absolute_error(y_home_val, home_pred_val)
mae_away = mean_absolute_error(y_away_val, away_pred_val)

rmse_home = root_mean_squared_error(y_home_val, home_pred_val)
rmse_away = root_mean_squared_error(y_away_val, away_pred_val)

print("MAE Home:", mae_home)
print("MAE Away:", mae_away)
print("RMSE Home:", rmse_home)
print("RMSE Away:", rmse_away)

MAE Home: 1.0341346263885498
MAE Away: 0.8509969711303711
RMSE Home: 1.3790521621704102
RMSE Away: 1.1128358840942383


In [11]:
def get_outcome(home, away):
    if home > away:
        return "H"
    elif home < away:
        return "A"
    else:
        return "D"

home_pred_round = np.round(home_pred_val).astype(int)
away_pred_round = np.round(away_pred_val).astype(int)

home_pred_round = np.clip(home_pred_round, 0, None)
away_pred_round = np.clip(away_pred_round, 0, None)

results_opt = pd.DataFrame({
    "home_real": y_home_val.values,
    "away_real": y_away_val.values,
    "home_pred": home_pred_round,
    "away_pred": away_pred_round
})

results_opt["real_outcome"] = results_opt.apply(
    lambda row: get_outcome(row["home_real"], row["away_real"]),
    axis=1
)

results_opt["pred_outcome"] = results_opt.apply(
    lambda row: get_outcome(row["home_pred"], row["away_pred"]),
    axis=1
)

accuracy_wdl = accuracy_score(
    results_opt["real_outcome"],
    results_opt["pred_outcome"]
)

exact_score = (
    (results_opt["home_real"] == results_opt["home_pred"]) &
    (results_opt["away_real"] == results_opt["away_pred"])
).mean()

print("Accuracy W-D-L:", accuracy_wdl)
print("Exact Score Accuracy:", exact_score)

Accuracy W-D-L: 0.563238512035011
Exact Score Accuracy: 0.1111597374179431


## Probando el test

In [12]:
home_pred_test = home_model_opt.predict(X_test)
away_pred_test = away_model_opt.predict(X_test)

In [13]:
mae_home_test = mean_absolute_error(y_home_test, home_pred_test)
mae_away_test = mean_absolute_error(y_away_test, away_pred_test)

rmse_home_test = root_mean_squared_error(y_home_test, home_pred_test)
rmse_away_test = root_mean_squared_error(y_away_test, away_pred_test)

print("MAE Home:", mae_home_test)
print("MAE Away:", mae_away_test)

print("RMSE Home:", rmse_home_test)
print("RMSE Away:", rmse_away_test)

MAE Home: 1.0558274984359741
MAE Away: 0.8728047609329224
RMSE Home: 1.3783122301101685
RMSE Away: 1.1830116510391235


In [14]:
home_pred_round_test = np.round(home_pred_test).astype(int)
away_pred_round_test = np.round(away_pred_test).astype(int)

home_pred_round_test = np.clip(home_pred_round_test, 0, None)
away_pred_round_test = np.clip(away_pred_round_test, 0, None)

In [15]:
results_test = pd.DataFrame({
    "home_real": y_home_test.values,
    "away_real": y_away_test.values,
    "home_pred": home_pred_round_test,
    "away_pred": away_pred_round_test
})

In [16]:
def get_outcome(home, away):
    if home > away:
        return "H"
    elif home < away:
        return "A"
    else:
        return "D"

In [17]:
results_test["real_outcome"] = results_test.apply(
    lambda row: get_outcome(row["home_real"], row["away_real"]),
    axis=1
)

results_test["pred_outcome"] = results_test.apply(
    lambda row: get_outcome(row["home_pred"], row["away_pred"]),
    axis=1
)

In [18]:

accuracy_wdl_test = accuracy_score(
    results_test["real_outcome"],
    results_test["pred_outcome"]
)

print("Accuracy WDL Test:", accuracy_wdl_test)

Accuracy WDL Test: 0.5727341964965728


In [19]:
exact_score_test = (
    (results_test["home_real"] == results_test["home_pred"])
    &
    (results_test["away_real"] == results_test["away_pred"])
).mean()

print("Exact Score Test:", exact_score_test)

Exact Score Test: 0.09291698400609291


In [20]:
test_metrics = pd.DataFrame({
    "Metric": [
        "MAE Home",
        "MAE Away",
        "RMSE Home",
        "RMSE Away",
        "Accuracy WDL",
        "Exact Score"
    ],
    "Value": [
        mae_home_test,
        mae_away_test,
        rmse_home_test,
        rmse_away_test,
        accuracy_wdl_test,
        exact_score_test
    ]
})

print(test_metrics)

         Metric     Value
0      MAE Home  1.055827
1      MAE Away  0.872805
2     RMSE Home  1.378312
3     RMSE Away  1.183012
4  Accuracy WDL  0.572734
5   Exact Score  0.092917


## Probando Mexico vs Sudafrica

In [29]:
selected_features_m4 = [
    "home_GF10",
    "home_GA10",
    "home_points10",
    "home_mean_competition10",

    "away_GF10",
    "away_GA10",
    "away_points10",
    "away_mean_competition10",

    "home_elo",
    "away_elo",
    "elo_diff",

    "h2h_home_wins",
    "h2h_away_wins",
    "h2h_draws",
    "h2h_matches",

    "neutral",
    "competition_level"
]

In [30]:
def get_latest_team_features(features_df, team, match_date):
    match_date = pd.to_datetime(match_date)

    team_matches = features_df[
        (
            (features_df["home_team"] == team) |
            (features_df["away_team"] == team)
        ) &
        (features_df["date"] < match_date)
    ].sort_values("date")

    if len(team_matches) == 0:
        raise ValueError(f"No hay partidos previos para {team}")

    last_match = team_matches.iloc[-1]

    if last_match["home_team"] == team:
        return {
            "GF10": last_match["home_GF10"],
            "GA10": last_match["home_GA10"],
            "points10": last_match["home_points10"],
            "mean_competition10": last_match["home_mean_competition10"],
            "elo": last_match["home_elo"]
        }
    else:
        return {
            "GF10": last_match["away_GF10"],
            "GA10": last_match["away_GA10"],
            "points10": last_match["away_points10"],
            "mean_competition10": last_match["away_mean_competition10"],
            "elo": last_match["away_elo"]
        }

In [31]:
def get_h2h_features(features_df, home_team, away_team, match_date):
    match_date = pd.to_datetime(match_date)

    h2h_matches = features_df[
        (
            (
                (features_df["home_team"] == home_team) &
                (features_df["away_team"] == away_team)
            )
            |
            (
                (features_df["home_team"] == away_team) &
                (features_df["away_team"] == home_team)
            )
        )
        &
        (features_df["date"] < match_date)
    ]

    h2h_home_wins = 0
    h2h_away_wins = 0
    h2h_draws = 0

    for _, row in h2h_matches.iterrows():
        if row["home_score"] == row["away_score"]:
            h2h_draws += 1

        elif row["home_score"] > row["away_score"]:
            winner = row["home_team"]

            if winner == home_team:
                h2h_home_wins += 1
            else:
                h2h_away_wins += 1

        else:
            winner = row["away_team"]

            if winner == home_team:
                h2h_home_wins += 1
            else:
                h2h_away_wins += 1

    return {
        "h2h_home_wins": h2h_home_wins,
        "h2h_away_wins": h2h_away_wins,
        "h2h_draws": h2h_draws,
        "h2h_matches": len(h2h_matches)
    }

In [32]:
def build_future_match_features(
    features_df,
    home_team,
    away_team,
    match_date,
    competition_level,
    neutral
):
    features_df = features_df.copy()
    features_df["date"] = pd.to_datetime(features_df["date"])

    home_f = get_latest_team_features(features_df, home_team, match_date)
    away_f = get_latest_team_features(features_df, away_team, match_date)

    h2h_f = get_h2h_features(features_df, home_team, away_team, match_date)

    row = {
        "home_GF10": home_f["GF10"],
        "home_GA10": home_f["GA10"],
        "home_points10": home_f["points10"],
        "home_mean_competition10": home_f["mean_competition10"],

        "away_GF10": away_f["GF10"],
        "away_GA10": away_f["GA10"],
        "away_points10": away_f["points10"],
        "away_mean_competition10": away_f["mean_competition10"],

        "home_elo": home_f["elo"],
        "away_elo": away_f["elo"],
        "elo_diff": home_f["elo"] - away_f["elo"],

        "h2h_home_wins": h2h_f["h2h_home_wins"],
        "h2h_away_wins": h2h_f["h2h_away_wins"],
        "h2h_draws": h2h_f["h2h_draws"],
        "h2h_matches": h2h_f["h2h_matches"],

        "neutral": neutral,
        "competition_level": competition_level
    }

    match_features = pd.DataFrame([row])

    return match_features[selected_features_m4]

In [33]:
def predict_match_score(
    features_df,
    home_team,
    away_team,
    match_date,
    competition_level,
    neutral,
    home_model,
    away_model
):
    match_features = build_future_match_features(
        features_df=features_df,
        home_team=home_team,
        away_team=away_team,
        match_date=match_date,
        competition_level=competition_level,
        neutral=neutral
    )

    home_pred_raw = home_model.predict(match_features)[0]
    away_pred_raw = away_model.predict(match_features)[0]

    home_pred = int(round(home_pred_raw))
    away_pred = int(round(away_pred_raw))

    home_pred = max(home_pred, 0)
    away_pred = max(away_pred, 0)

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_pred_raw": home_pred_raw,
        "away_pred_raw": away_pred_raw,
        "home_pred": home_pred,
        "away_pred": away_pred,
        "match_features": match_features
    }

In [52]:
prediction = predict_match_score(
    features_df=features_df,
    home_team="United States",
    away_team="Paraguay",
    match_date="2026-06-12",
    competition_level=6,
    neutral=0,
    home_model=home_model_opt,
    away_model=away_model_opt
)

print("Predicción cruda:")
print(prediction["home_team"], prediction["home_pred_raw"])
print(prediction["away_team"], prediction["away_pred_raw"])

print("\nMarcador predicho:")
print(
    f'{prediction["home_team"]} {prediction["home_pred"]} - '
    f'{prediction["away_pred"]} {prediction["away_team"]}'
)

Predicción cruda:
United States 1.8372792
Paraguay 0.815233

Marcador predicho:
United States 2 - 1 Paraguay


In [49]:
prediction = predict_match_score(
    features_df=features_df,
    home_team="United States",
    away_team="Paraguay",
    match_date="2026-06-12",
    competition_level=6,
    neutral=0,
    home_model=home_model_opt,
    away_model=away_model_opt
)

print("Predicción cruda:")
print(prediction["home_team"], prediction["home_pred_raw"])
print(prediction["away_team"], prediction["away_pred_raw"])

print("\nMarcador predicho:")
print(
    f'{prediction["home_team"]} {prediction["home_pred"]} - '
    f'{prediction["away_pred"]} {prediction["away_team"]}'
)

Predicción cruda:
United States 1.8372792
Paraguay 0.815233

Marcador predicho:
United States 2 - 1 Paraguay


In [53]:
import joblib

joblib.dump(home_model_opt, "models/home_model.pkl")
joblib.dump(away_model_opt, "models/away_model.pkl")

['models/away_model.pkl']

In [56]:
features_df.to_csv("data/features_final.csv", index=False)